# Problem

**State space.**
The states are encoded as integers
$$\mathcal{X} = {0,1,\dots,8} $$
corresponding to the grid in row-major order (top-left corner to bottom-right corner). State $8$ is the terminal goal state and is absorbing.

**Action space.**
Actions are represented as integers

$$ \mathcal{A} = {0,1,2,3} $$

where $0 =$ Up, $1 =$ Down, $2 =$ Left, $3 =$ Right.

**Transition dynamics.**
Transitions are deterministic. For any state $x$ and action $a$, the state index is first mapped to its grid coordinates $(r,c)$, the action moves the agent one step (unless this would leave the grid), and the result is mapped back to a state index. The goal state $8$ always transitions to itself.
The transition matrix $P$ has shape $(|\mathcal{X}||\mathcal{A}|) \times |\mathcal{X}| = 36 \times 9$, where each row corresponds to a pair $(x,a)$ and contains a one-hot vector indicating the unique next state:

$$P[(x,a),x'] = 1 \quad \text{iff } x' = \text{next\_state}(x,a)$$

and $0$ otherwise.

**Reward model.**
Thus $r(x,a) = 1$ only when $x$ is the goal state, and $0$ otherwise.

**Initial state.**
The initial state is fixed as $x_0 = 0$.

In [18]:
%load_ext autoreload
%autoreload 2

import os
import random
import numpy as np
import pandas as pd
import sys
import torch
from pathlib import Path

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
# Add project root to the Python path
sys.path.append(str(PROJECT_ROOT))

from fogas_torch import FOGASSolverBeta, FOGASOracleSolver, FOGASHyperOptimizer, FOGASEvaluator, PolicySolver, FOGASSolverBetaVectorized
from fogas_torch.dataset_collection.env_data_collector import EnvDataCollector


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Tabular Features

## Definition

### Description

**Feature Map**

We use a tabular one-hot feature representation over state–action pairs.
Each feature corresponds to a unique \((x,a)\) combination.

The feature map is defined as:
$$
\phi(x,a) \in \mathbb{R}^{36}
$$
with the ordering
$$
(x,a) \;\longrightarrow\; e_{\,x \cdot |\mathcal{A}| + a}
\quad |\mathcal{A}| = 4
$$

**Reward Weights**

The reward function is linear in the features:
$$
r(x,a) = \phi(x,a)^\top \omega.
$$

The weight vector $\omega \in \mathbb{R}^{36}$ is defined as:
$$
\omega_i =
\begin{cases}
1 & \text{if } i \in \{8\cdot 4,\, 8\cdot 4 + 1,\, 8\cdot 4 + 2,\, 8\cdot 4 + 3\}, \\
0 & \text{otherwise}.
\end{cases}
$$

This yields:
$$
r(x,a) =
\begin{cases}
1 & \text{if } x = 8 \;\;(\text{goal state}), \\
0 & \text{otherwise}.
\end{cases}
$$

**Transition Weights**

For each $x' \in \mathcal{X}$,
$$
\psi(x')_i =
\begin{cases}
1 & \text{if } i = 4x + a \text{ and } x' = \text{next\_state}(x,a), \\
0 & \text{otherwise}.
\end{cases}
$$

Equivalently, stacking $\psi(x')$ for all $x'$ recovers the tabular
transition matrix
$$
P \in \mathbb{R}^{36 \times 9}
$$
where each row $(x,a)$ contains a single 1 at the column corresponding
to $\text{next\_state}(x,a)$.

The terminal goal state $x = 8$ is absorbing, so for all actions $a$,
$$
\text{next\_state}(8,a) = 8
$$


## Testing

In [2]:
states = torch.arange(9, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)
N = len(states) # number of states
A = len(actions) # number of actions
gamma = 0.9

x_0 = 0 # fixed initial state
dataset_path = "/home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/grid3_problem.csv" # path where to download the dataset

goal_grid = 8   # absorbing terminal state

def phi(x, a):
    vec = torch.zeros(N * A, dtype=torch.float64)
    vec[int(x) * A + int(a)] = 1.0
    return vec

omega = torch.zeros(N * A, dtype=torch.float64)
omega[8 * A : 8 * A + 4] = 1.0

# Helper to convert index <-> (row, col)
def to_rc(s):  return divmod(int(s), 3)
def to_s(r,c): return r*3 + c

def next_state(s, a):
    s = int(s)
    a = int(a)
    if s == goal_grid:
        return goal_grid  # absorbing: if we reach that state we stay in that state

    r, c = to_rc(s)

    if a == 0:  # Up
        r = max(0, r-1)
    elif a == 1:  # Down
        r = min(2, r+1)
    elif a == 2:  # Left
        c = max(0, c-1)
    elif a == 3:  # Right
        c = min(2, c+1)

    return to_s(r, c)

def psi(xp):
    v = torch.zeros(N * A, dtype=torch.float64)
    for x in states:
        for a in actions:
            if next_state(x, a) == int(xp):
                v[int(x) * A + int(a)] = 1.0
    return v

mdp = PolicySolver(states=states, actions=actions, phi=phi, omega=omega, gamma=gamma, x0=x_0, psi=psi)


### Empirical

In [3]:
solver_e = FOGASSolverBeta(mdp=mdp, print_params=True, csv_path=dataset_path, seed=SEED, device=DEVICE)
evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(solver=solver_e,metric_callable=evaluator_e.get_metric("reward"))



Device: cpu
Dataset: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/grid3_problem.csv (n=150)

================ FOGAS PARAMETER SUMMARY ================

Basic Information
-----------------
Dataset size n:           150
Feature norm bound R:     1.0000
Num states N:             9
Num actions A:            4
Feature dim d:            36
Discount γ:               0.9
Confidence δ:             0.05

Theoretical Quantities
----------------------
T_min (theoretical):      138.8269278958555
T (iterations):                139

FOGAS Hyperparameters
---------------------
alpha:                        0.002354
rho:                            1389.672495
eta:                            0.000045
D_theta:                    18.973666
beta (ridge):             0.000200
D_pi (derived):           6.207977




Trying same optimal hyperparameters found for the oracle.

In [4]:
solver_e.run(alpha=1e-2, eta=1e-3, rho=0.5, tqdm_print=True, T=400)
evaluator_e.compare_value_functions()
evaluator_e.compare_final_rewards()

FOGAS: 100%|██████████| 400/400 [01:08<00:00,  5.80it/s]


========== VALUE FUNCTION COMPARISON ==========

State-wise V comparison:
State 0: V*(x) =  6.561000 | V^π(x) =  5.032937 | Δ = -1.528063e+00
State 1: V*(x) =  7.290000 | V^π(x) =  5.774958 | Δ = -1.515042e+00
State 2: V*(x) =  8.100000 | V^π(x) =  6.309177 | Δ = -1.790823e+00
State 3: V*(x) =  7.290000 | V^π(x) =  5.621093 | Δ = -1.668907e+00
State 4: V*(x) =  8.100000 | V^π(x) =  6.826726 | Δ = -1.273274e+00
State 5: V*(x) =  9.000000 | V^π(x) =  8.003590 | Δ = -9.964101e-01
State 6: V*(x) =  8.100000 | V^π(x) =  6.081505 | Δ = -2.018495e+00
State 7: V*(x) =  9.000000 | V^π(x) =  7.526774 | Δ = -1.473226e+00
State 8: V*(x) =  10.000000 | V^π(x) =  10.000000 | Δ = -1.243450e-14

Action-value Q comparison:
(x=0, a=0): Q*(x,a) =  5.904900 | Q^π(x,a) =  4.529643 | Δ = -1.375257e+00
(x=0, a=1): Q*(x,a) =  6.561000 | Q^π(x,a) =  5.058984 | Δ = -1.502016e+00
(x=0, a=2): Q*(x,a) =  5.904900 | Q^π(x,a) =  4.529643 | Δ = -1.375257e+00
(x=0, a=3): Q*(x,a) =  6.561000 | Q^π(x,a) =  5.197462 | Δ

In [5]:
evaluator_e.print_policy()

  State 0: π(a=0|s=0) = 0.11  π(a=1|s=0) = 0.42  π(a=2|s=0) = 0.05  π(a=3|s=0) = 0.42  --> best action: 1
  State 1: π(a=0|s=1) = 0.19  π(a=1|s=1) = 0.41  π(a=2|s=1) = 0.00  π(a=3|s=1) = 0.40  --> best action: 1
  State 2: π(a=0|s=2) = 0.26  π(a=1|s=2) = 0.43  π(a=2|s=2) = 0.04  π(a=3|s=2) = 0.28  --> best action: 1
  State 3: π(a=0|s=3) = 0.00  π(a=1|s=3) = 0.45  π(a=2|s=3) = 0.20  π(a=3|s=3) = 0.34  --> best action: 1
  State 4: π(a=0|s=4) = 0.05  π(a=1|s=4) = 0.44  π(a=2|s=4) = 0.04  π(a=3|s=4) = 0.47  --> best action: 3
  State 5: π(a=0|s=5) = 0.08  π(a=1|s=5) = 0.57  π(a=2|s=5) = 0.10  π(a=3|s=5) = 0.24  --> best action: 1
  State 6: π(a=0|s=6) = 0.03  π(a=1|s=6) = 0.25  π(a=2|s=6) = 0.24  π(a=3|s=6) = 0.48  --> best action: 3
  State 7: π(a=0|s=7) = 0.12  π(a=1|s=7) = 0.28  π(a=2|s=7) = 0.15  π(a=3|s=7) = 0.46  --> best action: 3
  State 8: π(a=0|s=8) = 0.35  π(a=1|s=8) = 0.35  π(a=2|s=8) = 0.15  π(a=3|s=8) = 0.15  --> best action: 0



# State-aggregation Features

**Feature Map**

We use a structured linear feature representation over state–action pairs that decomposes into row features, action features, and a terminal indicator.

The state space is a (3 \times 3) grid with
$$
\mathcal{X} = {0,\dots,8}, \quad \mathcal{A} = {0,1,2,3}.
$$

Let $ \text{row}(x) \in \{0,1,2\}$ denote the row index of state (x).
The feature map is defined as
$$
\phi(x,a) \in \mathbb{R}^{d}, \qquad d = 3 * 4 + 1 = 13.
$$

It has the following structure:
$$
\phi(x,a): \; \; (x,a) \;\longrightarrow\; e_{\; \text{row}(x) \; \cdot \; |\mathcal{A}|\; + \; a},
$$

Thus:

* The **first 3 coordinates** encode the row of the grid.
* The **next 4 coordinates** encode the chosen action.
* The **last coordinate** is a terminal-state indicator for the goal state (x=8).

**Reward Weights**

The reward function is linear in the features:
$$
r(x,a) = \phi(x,a)^\top \omega.
$$

The weight vector $\omega \in \mathbb{R}^{8}$ is defined as
$$
\omega =
\begin{bmatrix}
0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 0 \ 1
\end{bmatrix}.
$$

This yields:
$$
r(x,a) =
\begin{cases}
1 & \text{if } x = 8 \;\; (\text{goal state}), \\
0 & \text{otherwise}.
\end{cases}
$$

**Transition Model**

The transition matrix
$$
P \in \mathbb{R}^{(9\cdot 4) \times 9}
$$
is defined such that each row corresponding to $(x,a)$ contains a single 1 at column $x'$:

$$
P_{(x,a),x'} =
\begin{cases}
1 & \text{if } x' = \text{next\_state}(x,a), \\
0 & \text{otherwise}.
\end{cases}
$$


## Testing

In [19]:
states = torch.arange(9, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)
N = len(states) # number of states
A = len(actions) # number of actions
gamma = 0.9

x_0 = 0 # fixed initial state
dataset_path = "/home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/grid3_sa_gen.csv" 

goal = 8   # absorbing terminal state

# Helper to convert index <-> (row, col)
def to_rc(s):  return divmod(int(s), 3)
def to_s(r,c): return r*3 + c

d = (N // 3) * A + 1   # 3*4 + 1 = 13

def phi(state, action):
    r, _ = to_rc(state)
    f = torch.zeros(d, dtype=torch.float64)
    f[r * A + int(action)] = 1.0
    f[-1] = 1.0 if int(state) == goal else 0.0
    return f

omega = torch.zeros(d, dtype=torch.float64)
omega[d - 1] = 1.0

def next_state(s, a):
    s = int(s)
    a = int(a)
    if s == goal:
        return goal  # absorbing: if we reach that state we stay in that state

    r, c = to_rc(s)

    if a == 0:  # Up
        r = max(0, r-1)
    elif a == 1:  # Down
        r = min(2, r+1)
    elif a == 2:  # Left
        c = max(0, c-1)
    elif a == 3:  # Right
        c = min(2, c+1)

    return to_s(r, c)

P_grid = torch.zeros((N * A, N), dtype=torch.float64)
count = 0
for x in range(N):
    for a in range(A):
        xp = next_state(x, a)     # deterministic next state
        for xp_ in range(N):
            if xp_ == xp:
                P_grid[count, xp_] = 1.0
            else:
                P_grid[count, xp_] = 0.0
        count += 1

mdp = PolicySolver(states=states, actions=actions, phi=phi, omega=omega, gamma=gamma, x0=x_0, P=P_grid)

In [44]:
collector = EnvDataCollector(
    mdp=mdp,
    max_steps=30,
    terminal_states=[goal],
    seed=42,
)

df = collector.collect_mixed_dataset(
    policies=[mdp.pi_star, "random"],   # optimal + random
    proportions=[0.9, 0.1],            # 90% optimal, 10% random
    n_steps=1000,
    save_path=dataset_path,
    verbose=True,
    episode_based=True,                
)



  MIXED DATASET COLLECTION SUMMARY (TORCH)
Total transitions: 1000
Total episodes: 192
Mode: Episode-based

Policy Distribution:
  Policy 0:   707 steps (70.7%) | Target: 90.0% | Episodes: 177
  Policy 1:   293 steps (29.3%) | Target: 10.0% | Episodes: 16

✅ Mixed dataset saved to: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/grid3_sa_gen.csv


### Empirical

In [45]:
solver_e = FOGASSolverBetaVectorized(mdp=mdp, print_params=True, csv_path=dataset_path, seed=SEED, device=DEVICE)
evaluator_e = FOGASEvaluator(solver_e)
optimizer_e = FOGASHyperOptimizer(solver=solver_e,metric_callable=evaluator_e.get_metric("reward"))



Device: cpu
Dataset: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/grid3_sa_gen.csv (n=1000)

================ FOGAS PARAMETER SUMMARY ================

Basic Information
-----------------
Dataset size n:           1000
Feature norm bound R:     1.4142
Num states N:             9
Num actions A:            4
Feature dim d:            13
Discount γ:               0.9
Confidence δ:             0.05

Theoretical Quantities
----------------------
T_min (theoretical):      1851.025705278074
T (iterations):                1852

FOGAS Hyperparameters
---------------------
alpha:                        0.000759
rho:                            221.625558
eta:                            0.000024
D_theta:                    11.401754
beta (ridge):             0.000083
D_pi (derived):           16.023162




Trying same optimal hyperparameters found for the oracle.

In [54]:
solver_e.run(alpha=2e-3, eta=1e-4, rho=0.5, tqdm_print=True, T = 2000)
evaluator_e.compare_value_functions()
evaluator_e.compare_final_rewards()
evaluator_e.print_policy()

FOGAS:   0%|          | 0/2000 [00:00<?, ?it/s]

FOGAS: 100%|██████████| 2000/2000 [00:01<00:00, 1655.07it/s]


========== VALUE FUNCTION COMPARISON ==========

State-wise V comparison:
State 0: V*(x) =  6.561000 | V^π(x) =  3.371634 | Δ = -3.189366e+00
State 1: V*(x) =  7.290000 | V^π(x) =  4.547496 | Δ = -2.742504e+00
State 2: V*(x) =  8.100000 | V^π(x) =  6.491064 | Δ = -1.608936e+00
State 3: V*(x) =  7.290000 | V^π(x) =  3.737561 | Δ = -3.552439e+00
State 4: V*(x) =  8.100000 | V^π(x) =  5.173964 | Δ = -2.926036e+00
State 5: V*(x) =  9.000000 | V^π(x) =  7.861713 | Δ = -1.138287e+00
State 6: V*(x) =  8.100000 | V^π(x) =  4.086976 | Δ = -4.013024e+00
State 7: V*(x) =  9.000000 | V^π(x) =  5.853301 | Δ = -3.146699e+00
State 8: V*(x) =  10.000000 | V^π(x) =  10.000000 | Δ =  0.000000e+00

Action-value Q comparison:
(x=0, a=0): Q*(x,a) =  5.904900 | Q^π(x,a) =  3.034470 | Δ = -2.870430e+00
(x=0, a=1): Q*(x,a) =  6.561000 | Q^π(x,a) =  3.363805 | Δ = -3.197195e+00
(x=0, a=2): Q*(x,a) =  5.904900 | Q^π(x,a) =  3.034470 | Δ = -2.870430e+00
(x=0, a=3): Q*(x,a) =  6.561000 | Q^π(x,a) =  4.092746 | Δ